![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Disponibilización de modelos

En este notebook aprenderá a guardar un modelo y a disponibilizarlo como una API con la librería Flask. Una API (interfaz de programación de aplicaciones) es un conjunto de definiciones y protocolos que permiten que servicios, en este caso modelos, retornen resultados y respuestas sin necesidad de saber cómo están implementados.

## Instrucciones Generales:

Este notebook esta compuesto por dos secciones. En la primera secciónn, usted beberá entrenar y guardar (exportar) un modelo de random forest para predecir si una URL es phishing (fraudulenta) o no. En la segunda parte, usará el modelo entrenado y lo disponibilizara usando la libreria *Flask*. En el siguente paper puede conocer más detalles de la base de datos que usaremos y del problema: *A. Correa Bahnsen, E. C. Bohorquez, S. Villegas, J. Vargas, and F. A. Gonzalez, “Classifying phishing urls using recurrent neural networks,” in Electronic Crime Research (eCrime), 2017 APWG Symposium on. IEEE, 2017, pp. 1–8*. https://albahnsen.files.wordpress.com/2018/05/classifying-phishing-urls-using-recurrent-neural-networks_cameraready.pdf
  
Para realizar la actividad, solo siga las indicaciones asociadas a cada celda del notebook.


## Importar base de datos y librerías

In [ ]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Importación librerías
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import joblib

import os
os.chdir('..')

In [ ]:
# Carga de datos de archivos .csv
data = pd.read_csv('https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/datasets/phishing.csv')
data.head()

,url,phishing
0,http://www.subalipack.com/contact/images/sampl...,1
1,http://fasc.maximecapellot-gypsyjazz-ensemble....,1
2,http://theotheragency.com/confirmer/confirmer-...,1
3,http://aaalandscaping.com/components/com_smart...,1
4,http://paypal.com.confirm-key-21107316126168.s...,1


## Codificar variables categóricas
Relizar preprocesamiento de texto (URLs) para crear variables predictoras:

In [ ]:
# Creación de columnas binarias que indican si la URL contiene la palabra clave (keywords)
keywords = ['https', 'login', '.php', '.html', '@', 'sign']
for keyword in keywords:
    data['keyword_' + keyword] = data.url.str.contains(keyword).astype(int)

# Definición de la variable largo de la URL
data['lenght'] = data.url.str.len() - 2

# Definición de la variable largo del dominio de la URL
domain = data.url.str.split('/', expand=True).iloc[:, 2]
data['lenght_domain'] = domain.str.len()

# Definición de la variable binaria que indica si es IP
data['isIP'] = (domain.str.replace('.', '') * 1).str.isnumeric().astype(int)

# Definicón de la variable cuenta de 'com' en la URL
data['count_com'] = data.url.str.count('com')

data.head()

,url,phishing,keyword_https,keyword_login,keyword_.php,keyword_.html,keyword_@,keyword_sign,lenght,lenght_domain,isIP,count_com
0,http://www.subalipack.com/contact/images/sampl...,1,0,0,0,0,0,0,47,18,0,1
1,http://fasc.maximecapellot-gypsyjazz-ensemble....,1,0,0,0,0,0,0,73,41,0,0
2,http://theotheragency.com/confirmer/confirmer-...,1,0,0,0,0,0,0,92,18,0,1
3,http://aaalandscaping.com/components/com_smart...,1,0,0,0,0,0,0,172,18,0,3
4,http://paypal.com.confirm-key-21107316126168.s...,1,0,0,0,0,0,0,90,50,0,1


In [ ]:
# Separación de variables predictoras (X) y variable de interes (y)
X = data.drop(['url', 'phishing'], axis=1)
y = data.phishing

## Entrenar y guardar el modelo

In [ ]:
# Definición de modelo de clasificación Random Forest
clf = RandomForestClassifier(n_jobs=-1, n_estimators=100, max_depth=3)
cross_val_score(clf, X, y, cv=10)

array([0.75275, 0.758  , 0.7455 , 0.75425, 0.74825, 0.7615 , 0.75725,
       0.7545 , 0.756  , 0.7565 ])

In [ ]:
# Entrenamiento del modelo de clasificación Random Forest
clf.fit(X, y)

RandomForestClassifier(max_depth=3, n_jobs=-1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/MIAD_MLandNLP/models'
os.makedirs(path, exist_ok=True)

In [ ]:
# Exportar modelo a archivo binario .pkl
#joblib.dump(clf, 'model_deployment/phishing_clf.pkl', compress=3)
joblib.dump(clf, f'{path}/phishing_clf.pkl')

['/content/drive/MyDrive/MIAD_MLandNLP/models/phishing_clf.pkl']

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/MIAD_MLandNLP/models/phishing_clf.pkl'

clf_model = joblib.load(MODEL_PATH)

def transform(url):
    return 0

def predict_proba(url):
    features = transform(url)   # tu función de transformación
    return clf_model.predict_proba(features)

In [ ]:
predict_proba('http://www.vipturismolondres.com/com.br/?atendimento=Cliente&/LgSgkszm64/B8aNzHa8Aj.php')

array([[0.22662709, 0.77337291]])

In [ ]:
# Importar modelo y predicción
#from model_deployment.m09_model_deployment import predict_proba

# Predicción de probabilidad de que un link sea phishing
#predict_proba('http://www.vipturismolondres.com/com.br/?atendimento=Cliente&/LgSgkszm64/B8aNzHa8Aj.php')

## Disponibilizar modelo con Flask

Para esta sección del notebook instale las siguientes librerías *!pip install flask* y *!pip install flask_restplus*.

In [ ]:
# Instalación de librerías
!pip install flask
!pip install flask_restx

# Importación librerías
from flask import Flask
from flask_restx import Api, Resource, fields

In [ ]:
app = Flask(__name__)

# Definición API Flask
api = Api(
    app,
    version='1.0',
    title='Phishing Prediction API',
    description='Phishing Prediction API')

ns = api.namespace('predict',
     description='Phishing Classifier')

# Definición argumentos o parámetros de la API
parser = ns.parser()
parser.add_argument(
    'URL',
    type=str,
    required=True,
    help='URL to be analyzed',
    location='args')

resource_fields = api.model('Resource', {
    'result': fields.String,
})

In [ ]:
# Definición de la clase para disponibilización
@ns.route('/')
class PhishingApi(Resource):

    @ns.doc(parser=parser)
    @ns.marshal_with(resource_fields)
    def get(self):
        args = parser.parse_args()

        return {
         "result": predict_proba(args['URL'])
        }, 200

In [ ]:
# Ejecución de la aplicación que disponibiliza el modelo de manera local en el puerto 5000
app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


### Exponiendo tu API de Flask con ngrok

Para hacer tu API de Flask accesible desde tu máquina local o desde internet, usaremos `ngrok`. Sigue estos pasos:

In [ ]:
# 1. Instala pyngrok en tu entorno de Colab
!pip install pyngrok

Para usar ngrok, necesitarás un token de autenticación. Puedes obtener uno de forma gratuita registrándote en [ngrok.com](https://ngrok.com/).

In [ ]:
# 2. Introduce tu token de autenticación de ngrok
# **IMPORTANTE**: Reemplaza 'YOUR_NGROK_AUTH_TOKEN' con tu token real de ngrok.
# Es recomendable almacenar este token en los "Secrets" de Colab (el icono de la llave en el panel izquierdo).
# Si lo almacenas en Secrets como 'NGROK_AUTH_TOKEN', puedes usar:
# from google.colab import userdata
# NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

from pyngrok import ngrok

NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN" # <-- REEMPLAZA ESTO CON TU TOKEN DE NGROK
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
# 3. Inicia un túnel ngrok para el puerto 5000 (donde se ejecuta tu aplicación Flask)
# Asegúrate de que la celda `WBipEUwLeR8o` (la que ejecuta `app.run()`) esté activa y funcionando antes de ejecutar esta celda.

public_url = ngrok.connect(5000)
print(f"Tu API de Flask ahora es accesible públicamente en: {public_url}")

ERROR:pyngrok.process.ngrok:t=2026-04-19T17:09:36+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTH_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-19T17:09:36+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTH_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-19T17:09:36+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: The authtoken you specified does not look like a proper ngrok aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTH_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.

Una vez que ejecutes la celda anterior, ngrok te proporcionará una URL pública (por ejemplo, `https://abcdef12345.ngrok.io`). Podrás usar esa URL en tu navegador, seguida de `/predict/?URL=` y la URL que quieras analizar, para interactuar con tu API desde cualquier lugar.

Por ejemplo:
`https://abcdef12345.ngrok.io/predict/?URL=http://consultoriojuridico.co/pp/www.paypal.com/`

El modelo debe haber quedado disponibilizado en el puerto 5000. Para predecir la probabilidad de que una URL sea fraudulenta (phishing) copie en la barra de busqueda de su navegador la siguiente dirección (http://localhost:5000/predict/?URL=) y agregregue al final de esta la URL que desee precir. Por ejemplo, al copiar la URL http://localhost:5000/predict/?URL=http://consultoriojuridico.co/pp/www.paypal.com/, la API retornará la probabilidad de que la URL http://consultoriojuridico.co/pp/www.paypal.com/ sea phishing.